# Alzheimer's Disease Microglia scRNA-seq Pipeline
**Ritschel Research | Computational-First Invention Screen**

**Dataset:** GSE138852 — Human prefrontal cortex snRNA-seq
13,214 cells, 6 AD donors + 6 CT donors
Mathys et al., Nature 2019

**Goal:** Identify novel kinase inhibitors targeting
disease-associated microglia (DAM) in human AD brain

**Key findings:**
- Cluster 4 = microglia (P2RY12=0.992, CX3CR1=0.666)
- 620 DE genes; 8 targets; SYK dominant
- Named leads: fostamatinib (FDA-approved SYK),
  entospletinib, lanraplenib, cangrelor (P2RY12)

**Google Drive checkpoints:**
  My Drive/RitschelResearch/alzheimers/
  All h5ad, JSON, and CSV files saved there automatically.

**Data file:** GSE138852_counts.csv.gz must be uploaded
manually each new session via the Files panel (left sidebar).
Alternatively, save it once to your Drive folder and it
will be found automatically.

**Startup sequence after any restart:**
  1. Cell 1 — install (then restart session)
  2. Cell 2 — imports
  3. Cell 3 — mount Drive (authorize when prompted)
  4. Recovery Cell — restores from Drive automatically


In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Install dependencies
# After install: Runtime → Restart session,
# then run from Cell 2.
# ─────────────────────────────────────────────
!pip install scvi-tools scanpy anndata pandas numpy matplotlib seaborn \
             GEOparse requests tqdm scikit-learn chembl-webresource-client \
             leidenalg scipy -q

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Imports and shared utilities
# ─────────────────────────────────────────────
import os, json, time, warnings, requests, shutil
import scipy.sparse as sp
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
from tqdm import tqdm
from chembl_webresource_client.new_client import new_client
warnings.filterwarnings('ignore')
sc.settings.verbosity = 1

WORK_DIR   = '/content'
DRIVE_DIR  = None   # set after Cell 3
BATCH_SIZE = 50

def _drive_path(fname):
    return os.path.join(DRIVE_DIR, fname) if DRIVE_DIR else None

def save_checkpoint(data, fname):
    local = f'{WORK_DIR}/{fname}'
    with open(local, 'w') as f: json.dump(data, f)
    dp = _drive_path(fname)
    if dp: shutil.copy2(local, dp)

def load_checkpoint(fname):
    local = f'{WORK_DIR}/{fname}'
    dp    = _drive_path(fname)
    if not os.path.exists(local) and dp and os.path.exists(dp):
        shutil.copy2(dp, local)
        print(f'  Restored from Drive: {fname}')
    if os.path.exists(local):
        with open(local) as f: data = json.load(f)
        print(f'  Checkpoint found: {len(data)} entries.')
        return data
    print(f'  No checkpoint: {fname}')
    return {}

def save_h5ad(adata, fname):
    local = f'{WORK_DIR}/{fname}'
    adata.write(local)
    dp = _drive_path(fname)
    if dp:
        shutil.copy2(local, dp)
        print(f'  Saved + mirrored to Drive: {fname}')
    else:
        print(f'  Saved locally: {fname}')

def load_h5ad(fname):
    local = f'{WORK_DIR}/{fname}'
    dp    = _drive_path(fname)
    if not os.path.exists(local) and dp and os.path.exists(dp):
        print(f'  Restoring {fname} from Drive...')
        shutil.copy2(dp, local)
    if os.path.exists(local):
        return sc.read_h5ad(local)
    return None

def save_csv(df, fname):
    local = f'{WORK_DIR}/{fname}'
    df.to_csv(local, index=False)
    dp = _drive_path(fname)
    if dp: shutil.copy2(local, dp)

def save_png(fname):
    dp = _drive_path(fname)
    if dp: shutil.copy2(f'{WORK_DIR}/{fname}', dp)

print('Imports successful.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Mount Google Drive
# RUN THIS WHILE YOU ARE PRESENT to authorize.
# After authorization, all checkpoints save
# automatically to Drive every 50 compounds.
# ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/RitschelResearch/alzheimers'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted.')
print(f'Checkpoint directory: {DRIVE_DIR}')

existing = sorted(os.listdir(DRIVE_DIR))
if existing:
    print(f'\nExisting files on Drive ({len(existing)}):')
    for f in existing:
        size = os.path.getsize(os.path.join(DRIVE_DIR, f)) / 1e6
        print(f'  {f}  ({size:.1f} MB)')
else:
    print('Drive directory is empty — fresh start.')

## RECOVERY CELL — Run after any runtime restart
Automatically restores all checkpoints from Google Drive.

In [ ]:
# ─────────────────────────────────────────────
# RECOVERY CELL
# ─────────────────────────────────────────────
adata = None
recovery_stage = 'none'

for fname, stage, msg in [
    ('ad_clustered.h5ad', 'clustered', 'Continue from Cell 10 (marker dotplot)'),
    ('ad_scvi.h5ad',      'scvi',      'Continue from Cell 9 (clustering)'),
    ('ad_raw.h5ad',       'raw',       'Continue from Cell 5 (load check)'),
]:
    a = load_h5ad(fname)
    if a is not None:
        adata = a
        recovery_stage = stage
        print(f'Recovered [{stage}]: {adata.n_obs:,} cells')
        print(f'  {msg}')
        break

if adata is None:
    print('No h5ad on Drive. Upload GSE138852_counts.csv.gz then run Cell 4.')

# Restore JSON / CSV checkpoints from Drive
for fname in ['chembl_targets_checkpoint.json',
              'chembl_compounds_checkpoint.json',
              'novelty_checkpoint.json',
              'ad_DE_genes.csv']:
    local = f'{WORK_DIR}/{fname}'
    dp    = _drive_path(fname)
    if not os.path.exists(local) and dp and os.path.exists(dp):
        shutil.copy2(dp, local)
        size = os.path.getsize(local) / 1e6
        print(f'  Restored: {fname} ({size:.2f} MB)')

print(f'\nRecovery stage: {recovery_stage}')

## Phase 1 — Load Data

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Load GSE138852_counts.csv.gz
#
# File sources checked in order:
#   1. /content/GSE138852_counts.csv.gz  (manual upload)
#   2. /content/GSE138852/GSE138852_counts.csv.gz
#   3. Drive/RitschelResearch/alzheimers/GSE138852_counts.csv.gz
#      (save once to Drive to avoid re-uploading each session)
#
# TIP: After first upload, copy the file to your Drive folder:
#   import shutil
#   shutil.copy2('/content/GSE138852_counts.csv.gz', DRIVE_DIR)
# Then it loads automatically from Drive every session.
# ─────────────────────────────────────────────
if load_h5ad('ad_raw.h5ad') is not None:
    print('ad_raw.h5ad found — skip to Recovery Cell.')
else:
    csv_candidates = [
        '/content/GSE138852_counts.csv.gz',
        '/content/GSE138852/GSE138852_counts.csv.gz',
        f'{DRIVE_DIR}/GSE138852_counts.csv.gz' if DRIVE_DIR else None,
    ]
    csv_path = next((p for p in csv_candidates
                     if p and os.path.exists(p)), None)

    if csv_path is None:
        print('GSE138852_counts.csv.gz not found.')
        print('Upload it via the Files panel (left sidebar → upload arrow).')
        print('Or save it to your Drive folder once to avoid re-uploading.')
    else:
        print(f'Loading from: {csv_path}')
        df = pd.read_csv(csv_path, index_col=0)
        print(f'Shape (genes x cells): {df.shape}')

        X = sp.csr_matrix(df.values.T)
        adata = ad.AnnData(
            X=X,
            obs=pd.DataFrame(index=df.columns),
            var=pd.DataFrame(index=df.index)
        )
        adata.var_names_make_unique()

        # Extract donor and diagnosis from barcodes
        # Format: BARCODE_DonorID_BatchID
        # e.g. AAACCTGGTAGAAAGG_AD5_AD6
        # AD donors: AD1-AD6 | CT donors: Ct1-Ct6
        parts = adata.obs.index.str.rsplit('_', n=2)
        adata.obs['donor'] = [
            str(p[-2]) if len(p) >= 2 else 'unknown' for p in parts]
        adata.obs['batch'] = [
            str(p[-1]) if len(p) >= 1 else 'unknown' for p in parts]
        adata.obs['diagnosis'] = adata.obs['donor'].apply(
            lambda x: 'AD' if str(x).startswith('AD') else
                      'CT' if str(x).lower().startswith('ct') else 'unknown'
        )
        # Convert all obs to str for h5py compatibility
        for col in adata.obs.columns:
            adata.obs[col] = adata.obs[col].astype(str)

        print(f'Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes')
        print('Diagnosis:', adata.obs['diagnosis'].value_counts().to_dict())
        print('Donors:', sorted(adata.obs['donor'].unique()))

        save_h5ad(adata, 'ad_raw.h5ad')

        # Optionally copy CSV to Drive for future sessions
        if DRIVE_DIR and csv_path != f'{DRIVE_DIR}/GSE138852_counts.csv.gz':
            dest = f'{DRIVE_DIR}/GSE138852_counts.csv.gz'
            if not os.path.exists(dest):
                print('Copying CSV to Drive for future sessions...')
                shutil.copy2(csv_path, dest)
                print('Done — will load automatically next session.')

## Phase 2 — QC and Preprocessing

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Load raw checkpoint if needed
# ─────────────────────────────────────────────
if 'adata' not in dir() or adata is None:
    adata = load_h5ad('ad_raw.h5ad')
    if adata is None:
        print('No raw checkpoint — run Cell 4 first.')
    else:
        print(f'Loaded: {adata.n_obs:,} cells')
else:
    print(f'adata in memory: {adata.n_obs:,} cells')

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — QC metrics
# Expected: ~646 median genes, ~0.2% MT
# snRNA-seq: very low MT% (nuclei only)
# ─────────────────────────────────────────────
if load_h5ad('ad_scvi.h5ad') or load_h5ad('ad_clustered.h5ad'):
    print('scVI/clustering checkpoint found — skip.')
else:
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    print(f'Median genes/cell: {adata.obs["n_genes_by_counts"].median():.0f}')
    print(f'Median UMIs/cell:  {adata.obs["total_counts"].median():.0f}')
    print(f'Median MT%:        {adata.obs["pct_counts_mt"].median():.1f}%')
    sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
                 jitter=0.4, multi_panel=True)

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Filter, normalize, HVG
# snRNA-seq CSV: low UMI counts are normal.
# ─────────────────────────────────────────────
if load_h5ad('ad_scvi.h5ad') or load_h5ad('ad_clustered.h5ad'):
    print('Checkpoint found — skip.')
else:
    print(f'Before: {adata.n_obs:,} cells')
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=10)
    adata = adata[adata.obs['total_counts'] < 10000, :]
    adata = adata[adata.obs['pct_counts_mt'] < 5, :]
    print(f'After:  {adata.n_obs:,} cells')

    adata.layers['counts'] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.raw = adata

    sc.pp.highly_variable_genes(
        adata, n_top_genes=2000, subset=True,
        layer='counts', flavor='seurat_v3')
    print(f'HVGs: {adata.n_vars}')

## Phase 3 — scVI Training and Clustering

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — scVI setup and training
# batch_key='donor' (6 AD + 6 CT donors)
# GPU: ~2 min | Loss: ~431
# ─────────────────────────────────────────────
if load_h5ad('ad_scvi.h5ad') or load_h5ad('ad_clustered.h5ad'):
    print('Checkpoint found — skip to Recovery Cell, then Cell 9.')
else:
    batch_key = 'donor' if 'donor' in adata.obs.columns else None
    print(f'Batch key: {batch_key}')

    scvi.model.SCVI.setup_anndata(adata, layer='counts', batch_key=batch_key)
    model = scvi.model.SCVI(
        adata, n_layers=2, n_latent=30, gene_likelihood='nb')

    model.train(
        max_epochs=100,
        early_stopping=True,
        early_stopping_patience=10,
        plan_kwargs={'lr': 1e-3}
    )
    adata.obsm['X_scVI'] = model.get_latent_representation()
    print('scVI complete.')
    save_h5ad(adata, 'ad_scvi.h5ad')

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Clustering and UMAP
# Expected: 8 clusters at resolution 0.5
# Microglia confirmed = cluster 4
# ─────────────────────────────────────────────
if load_h5ad('ad_clustered.h5ad'):
    print('Checkpoint found — skip to Recovery Cell, then Cell 10.')
else:
    if 'X_scVI' not in adata.obsm:
        adata = load_h5ad('ad_scvi.h5ad')

    sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=15)
    for res in [0.3, 0.5, 0.8]:
        sc.tl.leiden(adata, resolution=res, key_added=f'leiden_{res}')
        print(f'  Resolution {res}: {adata.obs[f"leiden_{res}"].nunique()} clusters')

    adata.obs['leiden'] = adata.obs['leiden_0.5']
    sc.tl.umap(adata)
    sc.pl.umap(adata, color='leiden', legend_loc='on data',
               title='AD prefrontal cortex clusters (Leiden 0.5)')
    sc.pl.umap(adata, color='diagnosis', title='AD vs CT')

    save_h5ad(adata, 'ad_clustered.h5ad')

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — Identify and set microglia cluster
# Confirmed: cluster 4 is microglia
# P2RY12=0.992, CX3CR1=0.666, CSF1R=0.816
# ─────────────────────────────────────────────
microglia_markers = ['AIF1', 'CX3CR1', 'P2RY12', 'CSF1R',
                     'TMEM119', 'C1QA', 'C1QB', 'HEXB']
present_mg = [g for g in microglia_markers if g in adata.raw.var_names]
if present_mg:
    sc.pl.dotplot(adata, present_mg, groupby='leiden',
                  title='Microglia markers — cluster 4 confirmed')

MICROGLIA_CLUSTER = '4'
adata.obs['is_dam_cluster'] = (
    adata.obs['leiden'] == MICROGLIA_CLUSTER).astype(str)
n_mg = (adata.obs['is_dam_cluster'] == 'True').sum()
print(f'Microglia cluster {MICROGLIA_CLUSTER}: {n_mg:,} cells')
sc.pl.umap(adata, color='is_dam_cluster',
           title=f'Microglia cluster {MICROGLIA_CLUSTER}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 11 — Cell type marker dotplot
# ─────────────────────────────────────────────
BRAIN_MARKERS = {
    'Microglia':        ['AIF1', 'CX3CR1', 'P2RY12', 'CSF1R'],
    'Microglia (DAM)':  ['TREM2', 'AXL', 'SPP1', 'GPNMB', 'CLEC7A', 'APOE'],
    'Astrocytes':       ['GFAP', 'AQP4', 'SLC1A2'],
    'Oligodendrocytes': ['MBP', 'PLP1', 'MOG'],
    'OPCs':             ['PDGFRA', 'OLIG1'],
    'Excit. neurons':   ['NRGN', 'SLC17A7', 'CAMK2A'],
    'Inhib. neurons':   ['GAD1', 'GAD2'],
    'Endothelial':      ['PECAM1', 'FLT1'],
}
all_markers = [g for genes in BRAIN_MARKERS.values() for g in genes]
present = [g for g in all_markers if g in adata.raw.var_names]
print(f'Markers present: {len(present)}/{len(all_markers)}')
if present:
    sc.pl.dotplot(adata, present, groupby='leiden',
                  title='Brain cell type markers by cluster')

## Phase 4 — Differential Expression

In [ ]:
# ─────────────────────────────────────────────
# CELL 12 — Wilcoxon DE: microglia vs rest
# Expected: 620 DE genes
# Top: SYK, CLEC7A, CD86, PTPRC, DOCK8, SORL1
# ─────────────────────────────────────────────
de_csv_local = f'{WORK_DIR}/ad_DE_genes.csv'

if os.path.exists(de_csv_local):
    de_result = pd.read_csv(de_csv_local)
    print(f'Loaded DE genes: {len(de_result)}')
else:
    sc.tl.rank_genes_groups(
        adata, groupby='is_dam_cluster',
        groups=['True'], reference='False',
        method='wilcoxon', use_raw=True
    )
    de_result = sc.get.rank_genes_groups_df(
        adata, group='True', pval_cutoff=0.05, log2fc_min=0.5
    )
    save_csv(de_result, 'ad_DE_genes.csv')
    print(f'DE genes: {len(de_result)}')

print(de_result.head(20).to_string())

In [ ]:
# ─────────────────────────────────────────────
# CELL 13 — Volcano plot
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
de_plot = de_result.copy()
de_plot['-log10(padj)'] = -np.log10(de_plot['pvals_adj'].clip(1e-300))
de_plot['color'] = 'gray'
de_plot.loc[(de_plot['pvals_adj'] < 0.05) & (de_plot['logfoldchanges'] > 0.5), 'color'] = '#E24B4A'
de_plot.loc[(de_plot['pvals_adj'] < 0.05) & (de_plot['logfoldchanges'] < -0.5), 'color'] = '#378ADD'
ax.scatter(de_plot['logfoldchanges'], de_plot['-log10(padj)'],
           c=de_plot['color'], alpha=0.5, s=8)
label_genes = ['SYK', 'CLEC7A', 'CD86', 'P2RY12', 'CX3CR1',
                'DOCK8', 'SORL1', 'CSF1R', 'PTPRC']
for _, row in de_plot[de_plot['names'].isin(label_genes)].iterrows():
    ax.annotate(row['names'], (row['logfoldchanges'], row['-log10(padj)']),
                fontsize=8)
ax.axhline(-np.log10(0.05), color='gray', linestyle='--', alpha=0.5)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('log2 fold change')
ax.set_ylabel('-log10(adj p-value)')
ax.set_title('DE genes: microglia cluster 4 vs rest')
plt.tight_layout()
plt.savefig('ad_volcano.png', dpi=150)
save_png('ad_volcano.png')
plt.show()

## Phase 5 — ChEMBL Drug Screen

In [ ]:
# ─────────────────────────────────────────────
# CELL 14 — Map DE genes to ChEMBL targets
# Expected 8 targets: SYK, PTPRC, CLEC7A, CD86,
# GPR183, CSF3R, CX3CR1, P2RY12
# ─────────────────────────────────────────────
TOP_DE_GENES = 50
target_client   = new_client.target
activity_client = new_client.activity

top_de = de_result[
    (de_result['pvals_adj'] < 0.05) &
    (de_result['logfoldchanges'] > 0.5)
].sort_values('logfoldchanges', ascending=False).head(TOP_DE_GENES)

target_genes = top_de['names'].tolist()
print(f'Target genes: {len(target_genes)}')
print(target_genes[:20])

chembl_targets = load_checkpoint('chembl_targets_checkpoint.json')
remaining = [g for g in target_genes if g not in chembl_targets]

for gene in tqdm(remaining, desc='ChEMBL target lookup'):
    try:
        results = target_client.search(gene).filter(
            organism='Homo sapiens', target_type='SINGLE PROTEIN'
        ).only(['target_chembl_id', 'pref_name', 'target_type'])
        hits = list(results)
        chembl_targets[gene] = hits[0]['target_chembl_id'] if hits else None
    except:
        chembl_targets[gene] = None
    time.sleep(0.2)

save_checkpoint(chembl_targets, 'chembl_targets_checkpoint.json')

DAM_TARGETS = {
    'SYK', 'CLEC7A', 'CD86', 'PTPRC', 'CSF3R', 'GPR183',
    'TREM2', 'TYROBP', 'AXL', 'MERTK', 'TYRO3', 'CSF1R',
    'BTK', 'PIK3CG', 'PIK3CD', 'LYN', 'FYN',
    'SORL1', 'APOE', 'CD33', 'BIN1',
    'C1QA', 'C1QB', 'C1QC', 'ITGAM', 'ITGB2',
    'CX3CR1', 'CCR2', 'CXCR4', 'P2RY12',
    'MAPK14', 'RIPK1', 'JAK1', 'JAK2', 'STAT3',
}
mapped = {k: v for k, v in chembl_targets.items()
          if v and k in target_genes and k in DAM_TARGETS}
print(f'After DAM filter: {len(mapped)} targets')
for g, t in mapped.items(): print(f'  {g}: {t}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 15 — Retrieve ChEMBL inhibitors
# Expected: ~6,231 compound-target pairs
# ─────────────────────────────────────────────
compound_checkpoint = load_checkpoint('chembl_compounds_checkpoint.json')
all_compounds = []

for gene, target_id in tqdm(mapped.items(), desc='ChEMBL activity'):
    if target_id in compound_checkpoint:
        recs = [dict(r, gene_target=gene) for r in compound_checkpoint[target_id]]
        all_compounds.extend(recs)
        continue
    try:
        results = activity_client.filter(
            target_chembl_id=target_id,
            standard_type__in=['IC50', 'Ki', 'Kd'],
            standard_relation__in=['=', '<', '<='],
            assay_type='B'
        ).only(['molecule_chembl_id', 'canonical_smiles', 'pchembl_value',
                'molecule_pref_name', 'target_chembl_id'])
        hits = list(results)
        for h in hits: h['gene_target'] = gene
        compound_checkpoint[target_id] = hits
        all_compounds.extend(hits)
    except:
        compound_checkpoint[target_id] = []
    time.sleep(0.3)

save_checkpoint(compound_checkpoint, 'chembl_compounds_checkpoint.json')

df_compounds = pd.DataFrame(all_compounds).drop_duplicates()
df_compounds['pchembl_value'] = pd.to_numeric(
    df_compounds['pchembl_value'], errors='coerce')
df_compounds = df_compounds.dropna(subset=['pchembl_value', 'canonical_smiles'])
df_compounds = df_compounds[df_compounds['pchembl_value'] >= 5.0]
df_compounds = df_compounds.drop_duplicates(
    subset=['molecule_chembl_id', 'gene_target'], keep='first')
print(f'Compound-target pairs: {len(df_compounds)}')
print(f'Unique compounds: {df_compounds["molecule_chembl_id"].nunique()}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 16 — Priority scoring
# Staurosporine excluded (pan-kinase, toxic)
# ─────────────────────────────────────────────
compound_summary = df_compounds.groupby('molecule_chembl_id').agg(
    molecule_pref_name=('molecule_pref_name', 'first'),
    canonical_smiles=('canonical_smiles', 'first'),
    n_targets=('gene_target', 'nunique'),
    target_genes=('gene_target', lambda x: ', '.join(sorted(set(x)))),
    max_pchembl=('pchembl_value', 'max'),
    sum_pchembl=('pchembl_value', 'sum'),
).reset_index()

compound_summary['priority_score'] = (
    compound_summary['sum_pchembl'] * compound_summary['n_targets']
).round(1)
compound_summary = compound_summary.sort_values('priority_score', ascending=False)
compound_summary = compound_summary[
    compound_summary['molecule_pref_name'] != 'STAUROSPORINE']

print(f'Unique compounds: {len(compound_summary)}')
print('\nTop 20:')
print(compound_summary[[
    'molecule_chembl_id', 'molecule_pref_name', 'n_targets',
    'max_pchembl', 'priority_score', 'target_genes'
]].head(20).to_string())

## Phase 6 — USPTO Patent Novelty Check

In [ ]:
# ─────────────────────────────────────────────
# CELL 17 — USPTO novelty check (checkpoint/resume)
# Runtime: ~2 hours for ~6,000 compounds
# Saves to Drive every 50 compounds.
# Safe to leave running overnight.
# If interrupted: re-run this cell — it resumes
# automatically from the Drive checkpoint.
# ─────────────────────────────────────────────
USPTO_API = 'https://developer.uspto.gov/ibd-api/v1/patent/publication'

def check_uspto_novelty(compound_name, chembl_id):
    if not compound_name or str(compound_name) in ['nan', 'None', '']:
        compound_name = chembl_id
    queries = [
        f'{compound_name} Alzheimer',
        f'{compound_name} microglia neuroinflammation',
        f'{compound_name} TREM2 DAM dementia',
    ]
    total_hits = 0
    for q in queries:
        try:
            resp = requests.get(USPTO_API, params={
                'searchText': q, 'start': 0, 'rows': 5,
                'sources': 'US_PGPUB,USPAT'
            }, timeout=15)
            if resp.status_code == 200:
                total_hits += resp.json().get('response', {}).get('numFound', 0)
            elif resp.status_code == 429:
                print('  Rate limited — sleeping 60s'); time.sleep(60)
        except: pass
        time.sleep(0.3)
    return {
        'molecule_chembl_id': chembl_id,
        'patent_hits': total_hits,
        'novel_flag': 'NOVEL_ALL' if total_hits == 0 else 'PRIOR_ART'
    }

novelty_checkpoint = load_checkpoint('novelty_checkpoint.json')
already_checked    = set(novelty_checkpoint.keys())
remaining_novelty  = compound_summary[
    ~compound_summary['molecule_chembl_id'].isin(already_checked)]
print(f'Compounds to check: {len(remaining_novelty)} / {len(compound_summary)}')
if len(remaining_novelty) == 0:
    print('All checked — skip to Cell 18.')

batch_count = 0
for i, row in tqdm(remaining_novelty.iterrows(),
                   total=len(remaining_novelty), desc='USPTO'):
    cid = row['molecule_chembl_id']
    novelty_checkpoint[cid] = check_uspto_novelty(
        row.get('molecule_pref_name', ''), cid)
    batch_count += 1
    if batch_count % BATCH_SIZE == 0:
        save_checkpoint(novelty_checkpoint, 'novelty_checkpoint.json')
        print(f'  Saved at {batch_count} (local + Drive)')

save_checkpoint(novelty_checkpoint, 'novelty_checkpoint.json')
print(f'Complete: {len(novelty_checkpoint)} compounds checked')

df_novelty = pd.DataFrame(list(novelty_checkpoint.values()))
for col in ['patent_hits', 'novel_flag']:
    if col in compound_summary.columns:
        compound_summary = compound_summary.drop(columns=[col])
compound_summary = compound_summary.merge(
    df_novelty[['molecule_chembl_id', 'patent_hits', 'novel_flag']],
    on='molecule_chembl_id', how='left')
compound_summary['novel_flag'] = compound_summary['novel_flag'].fillna('UNCHECKED')

novel_count = (compound_summary['novel_flag'] == 'NOVEL_ALL').sum()
unchecked   = (compound_summary['novel_flag'] == 'UNCHECKED').sum()
print(f'NOVEL_ALL: {novel_count} / {len(compound_summary)}')
if unchecked > 0:
    print(f'UNCHECKED: {unchecked} — re-run this cell to continue from checkpoint.')

## Phase 7 — Final Ranking and Export

In [ ]:
# ─────────────────────────────────────────────
# CELL 18 — Final scoring and top 10
# ─────────────────────────────────────────────
compound_summary['novel_mult'] = compound_summary['novel_flag'].map(
    {'NOVEL_ALL': 2.0, 'PRIOR_ART': 1.0, 'UNCHECKED': 1.0})
compound_summary['final_score'] = (
    compound_summary['priority_score'] * compound_summary['novel_mult']
).round(1)
compound_summary = compound_summary.sort_values('final_score', ascending=False)

df_novel = compound_summary[
    compound_summary['novel_flag'] == 'NOVEL_ALL'].copy()

cols = ['molecule_chembl_id', 'molecule_pref_name', 'n_targets',
        'max_pchembl', 'novel_flag', 'final_score', 'target_genes']
print(f'NOVEL_ALL: {len(df_novel)}')
print('\nTop 10 NOVEL_ALL:')
print(df_novel[cols].head(10).to_string())

In [ ]:
# ─────────────────────────────────────────────
# CELL 19 — Bar chart
# ─────────────────────────────────────────────
top10 = df_novel.head(10).copy()
top10['label'] = top10.apply(
    lambda r: r['molecule_pref_name']
    if (r['molecule_pref_name'] is not None and
        str(r['molecule_pref_name']) not in ['nan', 'None', ''])
    else r['molecule_chembl_id'], axis=1
)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10['label'][::-1], top10['final_score'][::-1],
               color='#8E44AD', edgecolor='#4A235A', alpha=0.8)
ax.set_xlabel('Priority Score')
ax.set_title("Top 10 NOVEL_ALL Alzheimer's Microglia Candidates")
for bar, score in zip(bars, top10['final_score'][::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{score:.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('ad_top10.png', dpi=150)
save_png('ad_top10.png')
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 20 — Export all results to Drive
# ─────────────────────────────────────────────
save_csv(compound_summary, 'ad_all_candidates.csv')
save_csv(df_novel,         'ad_novel_candidates.csv')
save_csv(df_novel.head(10),'ad_top10_patent.csv')

print('Saved locally + to Drive:')
for f in ['ad_all_candidates.csv', 'ad_novel_candidates.csv',
          'ad_top10_patent.csv', 'ad_DE_genes.csv',
          'ad_volcano.png', 'ad_top10.png',
          'ad_raw.h5ad', 'ad_scvi.h5ad', 'ad_clustered.h5ad']:
    local = f'{WORK_DIR}/{f}'
    e = '\u2713' if os.path.exists(local) else '\u2717'
    print(f'  {e} {f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 21 — Summary report
# ─────────────────────────────────────────────
top1 = df_novel.iloc[0] if len(df_novel) > 0 else None

print('=' * 60)
print("ALZHEIMER'S MICROGLIA SCREEN — SUMMARY REPORT")
print('Ritschel Research')
print('=' * 60)
print(f'Dataset:              GSE138852')
print(f'Cells after QC:       {adata.n_obs:,}')
print(f'Clusters (res 0.5):   {adata.obs["leiden"].nunique()}')
print(f'Microglia cluster:    4 (P2RY12+, CX3CR1+, CSF1R+)')
print(f'DE genes:             {len(de_result)}')
print(f'ChEMBL targets:       {len(mapped)}')
print(f'Unique compounds:     {len(compound_summary)}')
print(f'NOVEL_ALL:            {len(df_novel)}')
print()
if top1 is not None:
    name = top1.get('molecule_pref_name', top1['molecule_chembl_id'])
    if str(name) in ['nan', 'None', '']: name = top1['molecule_chembl_id']
    print(f'TOP CANDIDATE: {name}')
    print(f'  ChEMBL ID:    {top1["molecule_chembl_id"]}')
    print(f'  Final Score:  {top1["final_score"]}')
    print(f'  pChEMBL:      {top1["max_pchembl"]:.2f}')
    print(f'  Target:       {top1["target_genes"]}')
    print(f'  Patent:       {top1["novel_flag"]}')
print()
print('Named leads:')
print('  fostamatinib (FDA-approved SYK inhibitor)')
print('  entospletinib, lanraplenib (SYK, clinical)')
print('  cangrelor, ticagrelor (P2RY12, FDA-approved)')
print()
print(f'Drive: {DRIVE_DIR}')
print()
print('Next steps:')
print('  1. Google Patents: fostamatinib + Alzheimer')
print('  2. Draft provisional patent specification')
print('  3. Archive on Zenodo (paper + spec)')
print('  4. git push github.com/glenritschel/alzheimers-scrna')